<a href="https://colab.research.google.com/github/Sabari19-adda/An-Efficient-SFSF-Knowledge-Distillation-Framework/blob/main/Evaluation_of_Teacher_models_with_CTD1%262.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === Minimal dual evaluation: old + new dataset ===
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from tensorflow.keras.models import load_model

# ===================== PATHS =====================
# Old dataset
X_PATH_OLD = "/content/drive/MyDrive/X.npy"
Y_PATH_OLD = "/content/drive/MyDrive/y.npy"

# New dataset
X_PATH_NEW = "/content/drive/MyDrive/X_new.npy"
Y_PATH_NEW = "/content/drive/MyDrive/y_new.npy"

# Model
MODEL_PATH = "/content/drive/MyDrive/Alzheimer_Models/InceptionV3_best_model.h5"
# =================================================


# ---------- Helpers ----------

def prepare_y(y):
    """Convert one-hot, strings, or ints → integer class labels."""
    if y.ndim == 2 and y.shape[1] > 1:
        return np.argmax(y, axis=1)
    elif y.dtype.type is np.str_ or y.dtype == object:
        le = LabelEncoder()
        return le.fit_transform(y.ravel())
    else:
        return y.astype(int).ravel()


def evaluate_dataset(X_path, Y_path, model, title="Dataset"):
    print(f"\n==================== Evaluating {title} ====================\n")

    # Load
    X = np.load(X_path, allow_pickle=True)
    y = np.load(Y_path, allow_pickle=True)

    y_true = prepare_y(y)

    # Predict
    y_pred_prob = model.predict(X, batch_size=32, verbose=1)

    if y_pred_prob.ndim == 2 and y_pred_prob.shape[1] > 1:
        y_pred = np.argmax(y_pred_prob, axis=1)
    else:
        y_pred = (y_pred_prob.ravel() >= 0.5).astype(int)

    # Report
    print("\nClassification Report:\n")
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    print("Confusion Matrix:\n", cm)

    # Per-class stats
    print("\nPer-class FPR, FNR, Sensitivity, Specificity:\n")
    fpr_list, fnr_list, sens_list, spec_list = [], [], [], []

    for i in range(cm.shape[0]):
        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP
        TN = cm.sum() - (TP + FP + FN)

        FPR = (FP / (FP + TN)) * 100 if (FP + TN) > 0 else 0
        FNR = (FN / (FN + TP)) * 100 if (FN + TP) > 0 else 0
        sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0

        fpr_list.append(FPR)
        fnr_list.append(FNR)
        sens_list.append(sensitivity)
        spec_list.append(specificity)

        print(f"Class {i}: FPR={FPR:.2f}%, FNR={FNR:.2f}%, Sensitivity={sensitivity:.4f}, Specificity={specificity:.4f}")

    # Summary
    print(f"\nAverage FPR: {np.mean(fpr_list):.2f}%")
    print(f"Average FNR: {np.mean(fnr_list):.2f}%")
    print(f"Macro Sensitivity: {np.mean(sens_list):.4f}")
    print(f"Macro Specificity: {np.mean(spec_list):.4f}")

    f1 = f1_score(y_true, y_pred, average="weighted") * 100
    acc = accuracy_score(y_true, y_pred) * 100

    print(f"\nWeighted F1 Score: {f1:.2f}%")
    print(f"Accuracy: {acc:.2f}%")

    print("\n============================================================\n")


# ---------- Load model ----------
print("Loading model...")
model = load_model(MODEL_PATH)
print("Model loaded successfully.\n")

# ---------- Run evaluations ----------
evaluate_dataset(X_PATH_OLD, Y_PATH_OLD, model, "OLD DATASET (X.npy / y.npy)")
evaluate_dataset(X_PATH_NEW, Y_PATH_NEW, model, "NEW DATASET (X_new.npy / y_new.npy)")


Mounted at /content/drive
Loading model...


Model loaded successfully.


==================== Evaluating OLD DATASET (X.npy / y.npy) ====================

47/47 ━━━━━━━━━━━━━━━━━━━━ 24s 276ms/step

Classification Report:

              precision    recall  f1-score   support

           0     0.9921    1.0000    0.9960       375
           1     1.0000    1.0000    1.0000       375
           2     0.9973    0.9920    0.9947       375
           3     0.9920    0.9893    0.9907       375

    accuracy                         0.9953      1500
   macro avg     0.9953    0.9953    0.9953      1500
weighted avg     0.9953    0.9953    0.9953      1500

Confusion Matrix:
 [[375   0   0   0]
 [  0 375   0   0]
 [  0   0 372   3]
 [  3   0   1 371]]

Per-class FPR, FNR, Sensitivity, Specificity:

Class 0: FPR=0.27%, FNR=0.00%, Sensitivity=1.0000, Specificity=0.9973
Class 1: FPR=0.00%, FNR=0.00%, Sensitivity=1.0000, Specificity=1.0000
Class 2: FPR=0.09%, FNR=0.80%, Sensitivity=0.9920, Specificity=0.9991
Class 3: FPR=0.27%, FNR=1.07%, Se

In [ ]:
# === Minimal dual evaluation: old + new dataset ===
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from tensorflow.keras.models import load_model

# ===================== PATHS =====================
# Old dataset
X_PATH_OLD = "/content/drive/MyDrive/X.npy"
Y_PATH_OLD = "/content/drive/MyDrive/y.npy"

# New dataset
X_PATH_NEW = "/content/drive/MyDrive/X_new.npy"
Y_PATH_NEW = "/content/drive/MyDrive/y_new.npy"

# Model
MODEL_PATH = "/content/drive/MyDrive/Alzheimer_Models/DenseNet121_best_model.h5"
# =================================================


# ---------- Helpers ----------

def prepare_y(y):
    """Convert one-hot, strings, or ints → integer class labels."""
    if y.ndim == 2 and y.shape[1] > 1:
        return np.argmax(y, axis=1)
    elif y.dtype.type is np.str_ or y.dtype == object:
        le = LabelEncoder()
        return le.fit_transform(y.ravel())
    else:
        return y.astype(int).ravel()


def evaluate_dataset(X_path, Y_path, model, title="Dataset"):
    print(f"\n==================== Evaluating {title} ====================\n")

    # Load
    X = np.load(X_path, allow_pickle=True)
    y = np.load(Y_path, allow_pickle=True)

    y_true = prepare_y(y)

    # Predict
    y_pred_prob = model.predict(X, batch_size=32, verbose=1)

    if y_pred_prob.ndim == 2 and y_pred_prob.shape[1] > 1:
        y_pred = np.argmax(y_pred_prob, axis=1)
    else:
        y_pred = (y_pred_prob.ravel() >= 0.5).astype(int)

    # Report
    print("\nClassification Report:\n")
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    print("Confusion Matrix:\n", cm)

    # Per-class stats
    print("\nPer-class FPR, FNR, Sensitivity, Specificity:\n")
    fpr_list, fnr_list, sens_list, spec_list = [], [], [], []

    for i in range(cm.shape[0]):
        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP
        TN = cm.sum() - (TP + FP + FN)

        FPR = (FP / (FP + TN)) * 100 if (FP + TN) > 0 else 0
        FNR = (FN / (FN + TP)) * 100 if (FN + TP) > 0 else 0
        sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0

        fpr_list.append(FPR)
        fnr_list.append(FNR)
        sens_list.append(sensitivity)
        spec_list.append(specificity)

        print(f"Class {i}: FPR={FPR:.2f}%, FNR={FNR:.2f}%, Sensitivity={sensitivity:.4f}, Specificity={specificity:.4f}")

    # Summary
    print(f"\nAverage FPR: {np.mean(fpr_list):.2f}%")
    print(f"Average FNR: {np.mean(fnr_list):.2f}%")
    print(f"Macro Sensitivity: {np.mean(sens_list):.4f}")
    print(f"Macro Specificity: {np.mean(spec_list):.4f}")

    f1 = f1_score(y_true, y_pred, average="weighted") * 100
    acc = accuracy_score(y_true, y_pred) * 100

    print(f"\nWeighted F1 Score: {f1:.2f}%")
    print(f"Accuracy: {acc:.2f}%")

    print("\n============================================================\n")


# ---------- Load model ----------
print("Loading model...")
model = load_model(MODEL_PATH)
print("Model loaded successfully.\n")

# ---------- Run evaluations ----------
evaluate_dataset(X_PATH_OLD, Y_PATH_OLD, model, "OLD DATASET (X.npy / y.npy)")
evaluate_dataset(X_PATH_NEW, Y_PATH_NEW, model, "NEW DATASET (X_new.npy / y_new.npy)")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading model...


Model loaded successfully.


==================== Evaluating OLD DATASET (X.npy / y.npy) ====================

47/47 ━━━━━━━━━━━━━━━━━━━━ 36s 447ms/step

Classification Report:

              precision    recall  f1-score   support

           0     0.9919    0.9787    0.9852       375
           1     0.9973    1.0000    0.9987       375
           2     0.9917    0.9520    0.9714       375
           3     0.9467    0.9947    0.9701       375

    accuracy                         0.9813      1500
   macro avg     0.9819    0.9813    0.9814      1500
weighted avg     0.9819    0.9813    0.9814      1500

Confusion Matrix:
 [[367   0   1   7]
 [  0 375   0   0]
 [  3   1 357  14]
 [  0   0   2 373]]

Per-class FPR, FNR, Sensitivity, Specificity:

Class 0: FPR=0.27%, FNR=2.13%, Sensitivity=0.9787, Specificity=0.9973
Class 1: FPR=0.09%, FNR=0.00%, Sensitivity=1.0000, Specificity=0.9991
Class 2: FPR=0.27%, FNR=4.80%, Sensitivity=0.9520, Specificity=0.9973
Class 3: FPR=1.87%, FNR=0.53%, Se

In [ ]:
# === Minimal dual evaluation: old + new dataset ===
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from tensorflow.keras.models import load_model

# ===================== PATHS =====================
# Old dataset
X_PATH_OLD = "/content/drive/MyDrive/X.npy"
Y_PATH_OLD = "/content/drive/MyDrive/y.npy"

# New dataset
X_PATH_NEW = "/content/drive/MyDrive/X_new.npy"
Y_PATH_NEW = "/content/drive/MyDrive/y_new.npy"

# Model
MODEL_PATH = "/content/drive/MyDrive/Alzheimer_Models/ResNet101V2_best_model.h5"
# =================================================


# ---------- Helpers ----------

def prepare_y(y):
    """Convert one-hot, strings, or ints → integer class labels."""
    if y.ndim == 2 and y.shape[1] > 1:
        return np.argmax(y, axis=1)
    elif y.dtype.type is np.str_ or y.dtype == object:
        le = LabelEncoder()
        return le.fit_transform(y.ravel())
    else:
        return y.astype(int).ravel()


def evaluate_dataset(X_path, Y_path, model, title="Dataset"):
    print(f"\n==================== Evaluating {title} ====================\n")

    # Load
    X = np.load(X_path, allow_pickle=True)
    y = np.load(Y_path, allow_pickle=True)

    y_true = prepare_y(y)

    # Predict
    y_pred_prob = model.predict(X, batch_size=32, verbose=1)

    if y_pred_prob.ndim == 2 and y_pred_prob.shape[1] > 1:
        y_pred = np.argmax(y_pred_prob, axis=1)
    else:
        y_pred = (y_pred_prob.ravel() >= 0.5).astype(int)

    # Report
    print("\nClassification Report:\n")
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    print("Confusion Matrix:\n", cm)

    # Per-class stats
    print("\nPer-class FPR, FNR, Sensitivity, Specificity:\n")
    fpr_list, fnr_list, sens_list, spec_list = [], [], [], []

    for i in range(cm.shape[0]):
        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP
        TN = cm.sum() - (TP + FP + FN)

        FPR = (FP / (FP + TN)) * 100 if (FP + TN) > 0 else 0
        FNR = (FN / (FN + TP)) * 100 if (FN + TP) > 0 else 0
        sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0

        fpr_list.append(FPR)
        fnr_list.append(FNR)
        sens_list.append(sensitivity)
        spec_list.append(specificity)

        print(f"Class {i}: FPR={FPR:.2f}%, FNR={FNR:.2f}%, Sensitivity={sensitivity:.4f}, Specificity={specificity:.4f}")

    # Summary
    print(f"\nAverage FPR: {np.mean(fpr_list):.2f}%")
    print(f"Average FNR: {np.mean(fnr_list):.2f}%")
    print(f"Macro Sensitivity: {np.mean(sens_list):.4f}")
    print(f"Macro Specificity: {np.mean(spec_list):.4f}")

    f1 = f1_score(y_true, y_pred, average="weighted") * 100
    acc = accuracy_score(y_true, y_pred) * 100

    print(f"\nWeighted F1 Score: {f1:.2f}%")
    print(f"Accuracy: {acc:.2f}%")

    print("\n============================================================\n")


# ---------- Load model ----------
print("Loading model...")
model = load_model(MODEL_PATH)
print("Model loaded successfully.\n")

# ---------- Run evaluations ----------
evaluate_dataset(X_PATH_OLD, Y_PATH_OLD, model, "OLD DATASET (X.npy / y.npy)")
evaluate_dataset(X_PATH_NEW, Y_PATH_NEW, model, "NEW DATASET (X_new.npy / y_new.npy)")


Mounted at /content/drive
Loading model...


Model loaded successfully.


==================== Evaluating OLD DATASET (X.npy / y.npy) ====================

47/47 ━━━━━━━━━━━━━━━━━━━━ 27s 337ms/step

Classification Report:

              precision    recall  f1-score   support

           0     0.9483    0.9787    0.9633       375
           1     0.9947    0.9973    0.9960       375
           2     0.9365    0.9440    0.9402       375
           3     0.9415    0.9013    0.9210       375

    accuracy                         0.9553      1500
   macro avg     0.9553    0.9553    0.9551      1500
weighted avg     0.9553    0.9553    0.9551      1500

Confusion Matrix:
 [[367   0   1   7]
 [  1 374   0   0]
 [  7   0 354  14]
 [ 12   2  23 338]]

Per-class FPR, FNR, Sensitivity, Specificity:

Class 0: FPR=1.78%, FNR=2.13%, Sensitivity=0.9787, Specificity=0.9822
Class 1: FPR=0.18%, FNR=0.27%, Sensitivity=0.9973, Specificity=0.9982
Class 2: FPR=2.13%, FNR=5.60%, Sensitivity=0.9440, Specificity=0.9787
Class 3: FPR=1.87%, FNR=9.87%, Se

In [ ]:
# === Minimal dual evaluation: old + new dataset ===
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from tensorflow.keras.models import load_model

# ===================== PATHS =====================
# Old dataset
X_PATH_OLD = "/content/drive/MyDrive/X.npy"
Y_PATH_OLD = "/content/drive/MyDrive/y.npy"

# New dataset
X_PATH_NEW = "/content/drive/MyDrive/X_new.npy"
Y_PATH_NEW = "/content/drive/MyDrive/y_new.npy"

# Model
MODEL_PATH = "/content/drive/MyDrive/Alzheimer_Models/InceptionResNetV2_best_model.keras"
# =================================================


# ---------- Helpers ----------

def prepare_y(y):
    """Convert one-hot, strings, or ints → integer class labels."""
    if y.ndim == 2 and y.shape[1] > 1:
        return np.argmax(y, axis=1)
    elif y.dtype.type is np.str_ or y.dtype == object:
        le = LabelEncoder()
        return le.fit_transform(y.ravel())
    else:
        return y.astype(int).ravel()


def evaluate_dataset(X_path, Y_path, model, title="Dataset"):
    print(f"\n==================== Evaluating {title} ====================\n")

    # Load
    X = np.load(X_path, allow_pickle=True)
    y = np.load(Y_path, allow_pickle=True)

    y_true = prepare_y(y)

    # Predict
    y_pred_prob = model.predict(X, batch_size=32, verbose=1)

    if y_pred_prob.ndim == 2 and y_pred_prob.shape[1] > 1:
        y_pred = np.argmax(y_pred_prob, axis=1)
    else:
        y_pred = (y_pred_prob.ravel() >= 0.5).astype(int)

    # Report
    print("\nClassification Report:\n")
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    print("Confusion Matrix:\n", cm)

    # Per-class stats
    print("\nPer-class FPR, FNR, Sensitivity, Specificity:\n")
    fpr_list, fnr_list, sens_list, spec_list = [], [], [], []

    for i in range(cm.shape[0]):
        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP
        TN = cm.sum() - (TP + FP + FN)

        FPR = (FP / (FP + TN)) * 100 if (FP + TN) > 0 else 0
        FNR = (FN / (FN + TP)) * 100 if (FN + TP) > 0 else 0
        sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0

        fpr_list.append(FPR)
        fnr_list.append(FNR)
        sens_list.append(sensitivity)
        spec_list.append(specificity)

        print(f"Class {i}: FPR={FPR:.2f}%, FNR={FNR:.2f}%, Sensitivity={sensitivity:.4f}, Specificity={specificity:.4f}")

    # Summary
    print(f"\nAverage FPR: {np.mean(fpr_list):.2f}%")
    print(f"Average FNR: {np.mean(fnr_list):.2f}%")
    print(f"Macro Sensitivity: {np.mean(sens_list):.4f}")
    print(f"Macro Specificity: {np.mean(spec_list):.4f}")

    f1 = f1_score(y_true, y_pred, average="weighted") * 100
    acc = accuracy_score(y_true, y_pred) * 100

    print(f"\nWeighted F1 Score: {f1:.2f}%")
    print(f"Accuracy: {acc:.2f}%")

    print("\n============================================================\n")


# ---------- Load model ----------
print("Loading model...")
model = load_model(MODEL_PATH)
print("Model loaded successfully.\n")

# ---------- Run evaluations ----------
evaluate_dataset(X_PATH_OLD, Y_PATH_OLD, model, "OLD DATASET (X.npy / y.npy)")
evaluate_dataset(X_PATH_NEW, Y_PATH_NEW, model, "NEW DATASET (X_new.npy / y_new.npy)")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading model...
Model loaded successfully.


==================== Evaluating OLD DATASET (X.npy / y.npy) ====================

47/47 ━━━━━━━━━━━━━━━━━━━━ 36s 462ms/step

Classification Report:

              precision    recall  f1-score   support

           0     0.9894    0.9947    0.9920       375
           1     1.0000    1.0000    1.0000       375
           2     0.9894    0.9920    0.9907       375
           3     0.9946    0.9867    0.9906       375

    accuracy                         0.9933      1500
   macro avg     0.9933    0.9933    0.9933      1500
weighted avg     0.9933    0.9933    0.9933      1500

Confusion Matrix:
 [[373   0   1   1]
 [  0 375   0   0]
 [  2   0 372   1]
 [  2   0   3 370]]

Per-class FPR, FNR, Sensitivity, Specificity:

Class 0: FPR=0.36%, FNR=0.53%, Sensitivity=0.9947, Specificity=0.9964
Class 1: FPR=0.00%, FNR=0.0